# Util Function

# Imports

In [1]:
from pesummary.io import read

from pycbc.inference.io import loadfile;
from pycbc import conversions as cvs;

import numpy as np
import matplotlib.pyplot as pp
import seaborn as sns

import pandas as pd

/home/chendawww/Software/anaconda3/envs/pycbc/lib/python3.12/site-packages/gwpy/time/__init__.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS


In [2]:
def augment_OGC_result(samples, approximant='SEOBNRv4PHM', f_ref=20) :
    # lalsuite里面只有SEOBNR的几个模型能算m_f, a_f
    samples = samples.add_fields(cvs.mass1_from_mchirp_q(samples['srcmchirp'], samples['q']), 'mass1')
    samples = samples.add_fields(cvs.mass2_from_mchirp_q(samples['srcmchirp'], samples['q']), 'mass2')
    samples = samples.add_fields(cvs.eta_from_mass1_mass2(samples['mass1'], samples['mass2']), 'eta')

    samples = samples.add_fields(cvs.mtotal_from_mass1_mass2(samples['mass1'], samples['mass2']), 'Mtot')
    samples = samples.add_fields(samples['srcmchirp'], 'Mchirp')
    # samples = samples.add_fields(cvs.eta_from_mass1_mass2(samples['mass1'], samples['mass2']), 'eta')
    samples = samples.add_fields(cvs.invq_from_mass1_mass2(samples['mass1'], samples['mass2']), 'invq')

    samples = samples.add_fields(cvs.chi_eff_from_spherical(samples['mass1'], samples['mass2'], samples['spin1_a'], samples['spin1_polar'], samples['spin2_a'], samples['spin2_polar']), 'chi_eff')
    samples = samples.add_fields(cvs.chi_p_from_spherical(samples['mass1'], samples['mass2'], samples['spin1_a'], samples['spin1_azimuthal'], samples['spin1_polar'], samples['spin2_a'],samples['spin1_azimuthal'], samples['spin2_polar']), 'chi_p')

    temp = {}
    temp['spin1_x'], temp['spin1_y'], temp['spin1_z'] = cvs._spherical_to_cartesian(samples['spin1_a'], samples['spin1_azimuthal'], samples['spin1_polar'])
    temp['spin2_x'], temp['spin2_y'], temp['spin2_z'] = cvs._spherical_to_cartesian(samples['spin2_a'], samples['spin2_azimuthal'], samples['spin2_polar'])

    samples = samples.add_fields(temp['spin1_x'], 'spin1_x')
    samples = samples.add_fields(temp['spin1_y'], 'spin1_y')
    samples = samples.add_fields(temp['spin1_z'], 'spin1_z')
    samples = samples.add_fields(temp['spin2_x'], 'spin2_x')
    samples = samples.add_fields(temp['spin2_y'], 'spin2_y')
    samples = samples.add_fields(temp['spin2_z'], 'spin2_z')

    # 这两个算得很慢
    # samples = samples.add_fields(cvs.final_mass_from_initial(samples['mass1'], samples['mass2'], samples['spin1_x'], samples['spin1_y'], samples['spin1_z'], samples['spin2_x'], samples['spin2_y'], samples['spin2_z'], approximant=approximant, f_ref=f_ref), 'm_f')
    # samples = samples.add_fields(cvs.final_spin_from_initial(samples['mass1'], samples['mass2'], samples['spin1_x'], samples['spin1_y'], samples['spin1_z'], samples['spin2_x'], samples['spin2_y'], samples['spin2_z'], approximant=approximant, f_ref=f_ref), 'a_f')

    return samples

# Read .hdf and Export .csv for Mathematica

In [3]:
posteriors_path = "/mnt/d/Documents/workspace/课题/workspace/Public/posteriors"
samples = {}

In [ ]:
# # 自己，OGC-4官方
# fp = loadfile('OGC4-GW150914.hdf', 'r')
# samples0 = fp.read_samples(fp['samples'].keys())

# # OGC-4官方
# fp = loadfile('GW150914_095045-PYCBC-POSTERIOR-IMRPhenomXPHM.hdf', 'r')
# samples1 = fp.read_samples(fp['samples'].keys())

# # OGC-3官方
# fp = loadfile('GW150914_095045-PYCBC-POSTERIOR-XPHM.hdf', 'r')
# samples2 = fp.read_samples(fp['samples'].keys())

# # 自己，OGC-4官方，no calibration
# fp = loadfile('GW150914_095045-PYCBC-POSTERIOR-XPHM.hdf', 'r')
# samples3 = fp.read_samples(fp['samples'].keys())


In [7]:
# 读取 hdf

fp = loadfile(posteriors_path + '/OGC/OGC4-GW150914_fdchi3.hdf', 'r')
samples['fdchi3'] = fp.read_samples(fp['samples'].keys())

fp = loadfile(posteriors_path + '/OGC/OGC4-GW150914_fdchi4.hdf', 'r')
samples['fdchi4'] = fp.read_samples(fp['samples'].keys())

fp = loadfile(posteriors_path + '/OGC/OGC4-GW150914_fdchi5l.hdf', 'r')
samples['fdchi5l'] = fp.read_samples(fp['samples'].keys())

fp = loadfile(posteriors_path + '/OGC/OGC4-GW150914_fdchi6l.hdf', 'r')
samples['fdchi6l'] = fp.read_samples(fp['samples'].keys())

In [4]:
fp = loadfile(posteriors_path + '/OGC/GW150914/OGC4-GW150914_cdchi3.hdf', 'r')
samples['cdchi3'] = fp.read_samples(fp['samples'].keys())

In [5]:
# 根据已有的参数样本补充更多组合参数
for key in samples.keys() :
    samples[key] = augment_OGC_result(samples[key])

In [6]:
# 将样本对象转为 DataFrame 对象
for key in samples.keys() :
    samples[key] = pd.DataFrame(data=np.transpose([samples[key][field] for field in samples[key].fields]), columns=samples[key].fields)


In [ ]:
csv_path = "/mnt/d/Documents/workspace/课题/workspace/Parameterized_PN_Tests/csv"

In [ ]:
csv_path = "/mnt/d/Documents/workspace/课题/workspace/Parameterized_PN_Tests/csv"
# 挑选参数输出为csv，以方便Mathematica操作

# samples["fdchi3"][["mass1","mass2","spin1_z","spin2_z","eta","Mchirp","dchi3"]].to_csv(csv_path+'/OGC4-GW150914_fdchi3.csv', index=False)
# samples["fdchi4"][["mass1","mass2","spin1_z","spin2_z","eta","Mchirp","dchi4"]].to_csv(csv_path+'/OGC4-GW150914_fdchi4.csv', index=False)
# samples["fdchi5l"][["mass1","mass2","spin1_z","spin2_z","eta","Mchirp","dchi5l"]].to_csv(csv_path+'/OGC4-GW150914_fdchi5l.csv', index=False)
# samples["fdchi6l"][["mass1","mass2","spin1_z","spin2_z","eta","Mchirp","dchi6l"]].to_csv(csv_path+'/OGC4-GW150914_fdchi6l.csv', index=False)
samples['cdchi3'][["mass1","mass2","spin1_z","spin2_z","eta","Mchirp","dchi3"]].to_csv(csv_path+'/GW150914/OGC4-GW150914_cdchi3.csv', index=False)